# MNIST MLP3 — old vs horizon-optimized comparison

Compares old SGD, old AdamW, old Muon, new AdamW, and new Muon with final metrics, validation-loss-selected metrics, matched-seed contrasts, convergence, full plots, zoomed plots, and WeightWatcher spectral diagnostics.

In [ ]:
from pathlib import Path
import json
import math
import os
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

ROOT = None
for path in [Path.cwd(), *Path.cwd().parents]:
    candidate = path / 'baseline'
    if (candidate / 'rg_baselines').is_dir():
        ROOT = candidate
        break
    if (path / 'rg_baselines').is_dir():
        ROOT = path
        break
if ROOT is None:
    raise RuntimeError('Run from a current clone of CalculatedContent/rg_optimizers.')
ROOT = ROOT.resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from rg_baselines import DEFAULT_BASELINE_SEEDS, MNIST_REFERENCE_INITIALIZATION, MNIST_REFERENCE_RECIPE_VERSION, MNIST_REFERENCE_SUITE_SLUG
from rg_baselines.statistics import student_t_critical_95
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)
BASE_RUN_ROOT = Path(os.environ.get('RG_BASELINE_RUN_ROOT', ROOT / 'runs')).expanduser().resolve()
RUN_ROOT = BASE_RUN_ROOT / MNIST_REFERENCE_SUITE_SLUG
OUTPUT_DIR = RUN_ROOT / 'horizon_optimized_comparison'
PLOT_DIR = OUTPUT_DIR / 'plots'
ZOOM_DIR = PLOT_DIR / 'zoomed'
for directory in (OUTPUT_DIR, PLOT_DIR, ZOOM_DIR):
    directory.mkdir(parents=True, exist_ok=True)
ARM_SPECS = [
    {'arm': 'old_sgd', 'label': 'old SGD + Nesterov', 'family': 'sgd_momentum', 'variant': 'old', 'directory': RUN_ROOT / 'sgd_momentum', 'color': '#0072B2', 'linestyle': '-'},
    {'arm': 'old_adamw', 'label': 'old AdamW', 'family': 'adamw', 'variant': 'old', 'directory': RUN_ROOT / 'adamw', 'color': '#D55E00', 'linestyle': ':'},
    {'arm': 'new_adamw', 'label': 'new AdamW, horizon optimized', 'family': 'adamw', 'variant': 'horizon_optimized', 'directory': RUN_ROOT / 'adamw_horizon_optimized', 'color': '#E69F00', 'linestyle': '-'},
    {'arm': 'old_muon', 'label': 'old Muon + aux AdamW', 'family': 'sgd_momentum_muon', 'variant': 'old', 'directory': RUN_ROOT / 'sgd_momentum_muon', 'color': '#009E73', 'linestyle': ':'},
    {'arm': 'new_muon', 'label': 'new Muon, horizon optimized', 'family': 'sgd_momentum_muon', 'variant': 'horizon_optimized', 'directory': RUN_ROOT / 'sgd_momentum_muon_horizon_optimized', 'color': '#56B4E9', 'linestyle': '-'},
]
ARM_ORDER = tuple(spec['arm'] for spec in ARM_SPECS)
ARM_LABELS = {spec['arm']: spec['label'] for spec in ARM_SPECS}
SEEDS = tuple(DEFAULT_BASELINE_SEEDS)
LAYER_ORDER = ('fc1', 'fc2', 'fc3')
print('versioned suite root:', RUN_ROOT)
display(pd.DataFrame(ARM_SPECS).drop(columns=['color', 'linestyle']))


In [ ]:
REQUIRED_FILES = ('performance_by_epoch_and_seed.csv', 'spectral_metrics_by_epoch_layer_and_seed.csv', 'replicate_manifest.json')
def load_arm(spec):
    directory = Path(spec['directory'])
    missing = [directory / name for name in REQUIRED_FILES if not (directory / name).is_file()]
    if missing:
        raise FileNotFoundError('Run the notebook for ' + spec['label'] + ' before comparison:' + chr(10) + chr(10).join(map(str, missing)))
    manifest = json.loads((directory / 'replicate_manifest.json').read_text(encoding='utf-8'))
    cfg = manifest['config_template']
    if cfg['recipe_version'] != MNIST_REFERENCE_RECIPE_VERSION or cfg['initialization'] != MNIST_REFERENCE_INITIALIZATION:
        raise RuntimeError('wrong recipe or initialization for ' + spec['label'])
    if tuple(int(seed) for seed in manifest['seeds']) != SEEDS:
        raise RuntimeError('seed tuple mismatch for ' + spec['label'])
    perf = pd.read_csv(directory / 'performance_by_epoch_and_seed.csv')
    specm = pd.read_csv(directory / 'spectral_metrics_by_epoch_layer_and_seed.csv')
    for frame in (perf, specm):
        frame.insert(0, 'arm', spec['arm'])
        frame.insert(1, 'arm_label', spec['label'])
        frame.insert(2, 'family', spec['family'])
        frame.insert(3, 'variant', spec['variant'])
    return manifest, perf, specm
manifests = {}
perf_frames = []
spec_frames = []
for spec in ARM_SPECS:
    manifest, perf, specm = load_arm(spec)
    manifests[spec['arm']] = manifest
    perf_frames.append(perf)
    spec_frames.append(specm)
performance = pd.concat(perf_frames, ignore_index=True, sort=False)
spectral_metrics = pd.concat(spec_frames, ignore_index=True, sort=False)
performance['test_perplexity'] = np.exp(performance['test_loss'].astype(float))
performance['validation_perplexity'] = np.exp(performance['validation_loss'].astype(float))
config_table = pd.DataFrame([{'arm': arm, 'arm_label': ARM_LABELS[arm], **manifest['config_template']} for arm, manifest in manifests.items()])
config_table.to_csv(OUTPUT_DIR / 'arm_configurations.csv', index=False)
display(config_table)
print('performance rows:', len(performance), 'spectral rows:', len(spectral_metrics))


In [ ]:
def summarize(frame, groups, metrics):
    rows = []
    for keys, group in frame.groupby(list(groups), dropna=False, sort=True):
        if not isinstance(keys, tuple):
            keys = (keys,)
        base = dict(zip(groups, keys, strict=True))
        for metric in metrics:
            if metric not in group.columns:
                continue
            values = pd.to_numeric(group[metric], errors='coerce').to_numpy(dtype=float)
            values = values[np.isfinite(values)]
            if len(values) == 0:
                continue
            n = int(len(values))
            mean = float(values.mean())
            std = float(values.std(ddof=1)) if n > 1 else 0.0
            sem = std / math.sqrt(n) if n > 1 else 0.0
            half = float(student_t_critical_95(n)) * sem if n > 1 else 0.0
            rows.append({**base, 'metric': metric, 'n': n, 'mean': mean, 'std': std, 'sem': sem, 'ci_half_width': half, 'ci_low': mean - half, 'ci_high': mean + half, 'minimum': float(values.min()), 'maximum': float(values.max())})
    return pd.DataFrame(rows)
PERF_METRICS = ['train_loss', 'validation_loss', 'test_loss', 'train_accuracy', 'validation_accuracy', 'test_accuracy', 'validation_loss_gap', 'test_loss_gap', 'validation_accuracy_gap', 'test_accuracy_gap', 'primary_lr', 'auxiliary_lr', 'parameter_l2_norm', 'mean_gradient_norm_before_clip', 'max_gradient_norm_before_clip']
performance_summary = summarize(performance, ('arm', 'arm_label', 'family', 'variant', 'epoch'), PERF_METRICS)
def terminal_rows(frame):
    rows = []
    for (arm, seed), run in frame.groupby(['arm', 'seed'], sort=True):
        ordered = run.sort_values('epoch')
        final = ordered.iloc[-1].copy()
        final['checkpoint'] = 'final'
        rows.append(final)
        selected = ordered.sort_values(['validation_loss', 'epoch'], ascending=[True, True]).iloc[0].copy()
        selected['checkpoint'] = 'validation_selected'
        rows.append(selected)
    return pd.DataFrame(rows).reset_index(drop=True)
terminal_by_seed = terminal_rows(performance)
terminal_summary = summarize(terminal_by_seed, ('arm', 'arm_label', 'family', 'variant', 'checkpoint'), PERF_METRICS)
terminal_by_seed.to_csv(OUTPUT_DIR / 'terminal_by_seed.csv', index=False)
terminal_summary.to_csv(OUTPUT_DIR / 'terminal_summary_95ci.csv', index=False)
display(terminal_by_seed.sort_values(['checkpoint', 'family', 'variant', 'seed']))
display(terminal_summary[terminal_summary['metric'].isin(['test_accuracy', 'test_loss', 'validation_loss', 'parameter_l2_norm'])].sort_values(['checkpoint', 'metric', 'family', 'variant']))


In [ ]:
def metric_row(summary, arm, checkpoint, metric):
    rows = summary.loc[summary['arm'].eq(arm) & summary['checkpoint'].eq(checkpoint) & summary['metric'].eq(metric)]
    if rows.empty:
        raise KeyError((arm, checkpoint, metric))
    return rows.iloc[0]
rank_specs = [('final', 'test_accuracy', 'max'), ('final', 'test_loss', 'min'), ('validation_selected', 'test_accuracy', 'max'), ('validation_selected', 'test_loss', 'min'), ('validation_selected', 'validation_loss', 'min')]
rank_rows = []
for checkpoint, metric, direction in rank_specs:
    vals = []
    for arm in ARM_ORDER:
        row = metric_row(terminal_summary, arm, checkpoint, metric)
        vals.append((arm, float(row['mean']), float(row['ci_half_width'])))
    vals.sort(key=lambda item: item[1], reverse=(direction == 'max'))
    for rank, (arm, mean, half) in enumerate(vals, start=1):
        rank_rows.append({'checkpoint': checkpoint, 'metric': metric, 'direction': direction, 'rank': rank, 'arm': arm, 'arm_label': ARM_LABELS[arm], 'mean': mean, 'ci_half_width': half})
rank_table = pd.DataFrame(rank_rows)
display(rank_table)


In [ ]:
PAIR_METRICS = ['test_accuracy', 'test_loss', 'validation_loss', 'parameter_l2_norm']
CONTRASTS = [('new_adamw', 'old_adamw'), ('new_muon', 'old_muon'), ('new_adamw', 'old_sgd'), ('new_muon', 'old_sgd'), ('new_muon', 'new_adamw')]
rows = []
for checkpoint in ('final', 'validation_selected'):
    selected = terminal_by_seed.loc[terminal_by_seed['checkpoint'].eq(checkpoint)]
    for arm_a, arm_b in CONTRASTS:
        for metric in PAIR_METRICS:
            a = selected.loc[selected['arm'].eq(arm_a), ['seed', metric]].rename(columns={metric: 'value_a'})
            b = selected.loc[selected['arm'].eq(arm_b), ['seed', metric]].rename(columns={metric: 'value_b'})
            paired = a.merge(b, on='seed', validate='one_to_one').sort_values('seed')
            values = (paired['value_a'] - paired['value_b']).to_numpy(dtype=float)
            n = len(values)
            mean = float(values.mean())
            std = float(values.std(ddof=1)) if n > 1 else 0.0
            half = float(student_t_critical_95(n)) * std / math.sqrt(n) if n > 1 else 0.0
            rows.append({'checkpoint': checkpoint, 'contrast': ARM_LABELS[arm_a] + ' - ' + ARM_LABELS[arm_b], 'metric': metric, 'n': n, 'mean_difference': mean, 'ci_half_width': half, 'ci_low': mean - half, 'ci_high': mean + half})
paired_differences = pd.DataFrame(rows)
paired_differences.to_csv(OUTPUT_DIR / 'paired_old_new_differences_95ci.csv', index=False)
display(paired_differences.sort_values(['checkpoint', 'metric', 'contrast']))


In [ ]:
def plot_perf(metric, percent=False, zoom_start=None):
    data = performance.copy()
    summary = performance_summary.copy()
    if zoom_start is not None:
        data = data.loc[data['epoch'].astype(int).ge(zoom_start)]
        summary = summary.loc[summary['epoch'].astype(int).ge(zoom_start)]
    fig, ax = plt.subplots(figsize=(11, 6))
    for spec in ARM_SPECS:
        raw = data.loc[data['arm'].eq(spec['arm'])]
        for seed, run in raw.groupby('seed'):
            values = pd.to_numeric(run[metric], errors='coerce').to_numpy(dtype=float)
            if percent:
                values = 100.0 * values
            ax.plot(run['epoch'], values, color=spec['color'], linestyle=spec['linestyle'], linewidth=0.8, alpha=0.15)
        rows = summary.loc[summary['arm'].eq(spec['arm']) & summary['metric'].eq(metric)].sort_values('epoch')
        if rows.empty:
            continue
        x = rows['epoch'].astype(float).to_numpy()
        mean = rows['mean'].astype(float).to_numpy()
        low = rows['ci_low'].astype(float).to_numpy()
        high = rows['ci_high'].astype(float).to_numpy()
        half = rows['ci_half_width'].astype(float).to_numpy()
        if percent:
            mean, low, high, half = [100.0 * v for v in (mean, low, high, half)]
        ax.plot(x, mean, color=spec['color'], linestyle=spec['linestyle'], linewidth=2.4, label=spec['label'])
        ax.fill_between(x, low, high, color=spec['color'], alpha=0.10)
        step = max(1, len(x) // 12)
        ax.errorbar(x[::step], mean[::step], yerr=half[::step], fmt='none', ecolor=spec['color'], elinewidth=0.8, capsize=2.0)
    ax.set(xlabel='Epoch', ylabel=metric + (' (%)' if percent else ''), title=metric if zoom_start is None else f'{metric} from epoch {zoom_start}')
    ax.grid(alpha=0.22)
    ax.legend(frameon=False, fontsize=8)
    fig.tight_layout()
    out = (ZOOM_DIR if zoom_start is not None else PLOT_DIR) / ((f'zoom_epoch{zoom_start}_' if zoom_start is not None else 'full_') + metric + '.png')
    fig.savefig(out, dpi=180, bbox_inches='tight')
    plt.show()
for metric, percent in [('test_accuracy', True), ('validation_accuracy', True), ('test_loss', False), ('validation_loss', False), ('train_loss', False), ('test_loss_gap', False), ('parameter_l2_norm', False), ('primary_lr', False), ('auxiliary_lr', False)]:
    if metric in performance.columns and performance[metric].notna().any():
        plot_perf(metric, percent=percent)
        plot_perf(metric, percent=percent, zoom_start=3)


In [ ]:
valid = spectral_metrics.loc[spectral_metrics['status'].astype(str).eq('ok')].copy()
SPEC_METRICS = ['alpha', 'num_traps', 'ERG_gap', 'm_midpoint', 'trace_log_midpoint_per_eval', 'stable_rank']
spectral_summary = summarize(valid, ('arm', 'arm_label', 'family', 'variant', 'layer', 'epoch'), SPEC_METRICS)
display(spectral_summary.sort_values(['metric', 'family', 'variant', 'layer', 'epoch']))
def plot_spec(metric, zoom_start=None, ylims=None, ref=None):
    data = valid.copy()
    summary = spectral_summary.copy()
    if zoom_start is not None:
        data = data.loc[data['epoch'].astype(int).ge(zoom_start)]
        summary = summary.loc[summary['epoch'].astype(int).ge(zoom_start)]
    fig, axes = plt.subplots(1, len(LAYER_ORDER), figsize=(17, 5), sharex=True)
    for ax, layer in zip(axes, LAYER_ORDER, strict=True):
        for spec in ARM_SPECS:
            raw = data.loc[data['arm'].eq(spec['arm']) & data['layer'].astype(str).eq(layer)]
            for seed, run in raw.groupby('seed'):
                ax.plot(run['epoch'], run[metric], color=spec['color'], linestyle=spec['linestyle'], linewidth=0.75, alpha=0.12)
            rows = summary.loc[summary['arm'].eq(spec['arm']) & summary['layer'].astype(str).eq(layer) & summary['metric'].eq(metric)].sort_values('epoch')
            if rows.empty:
                continue
            ax.plot(rows['epoch'], rows['mean'], color=spec['color'], linestyle=spec['linestyle'], linewidth=2.2, label=spec['label'])
            ax.fill_between(rows['epoch'], rows['ci_low'], rows['ci_high'], color=spec['color'], alpha=0.09)
        if ref is not None:
            ax.axhline(ref, color='black', linestyle='--', linewidth=1.0, alpha=0.65)
        if ylims and layer in ylims:
            ax.set_ylim(*ylims[layer])
        ax.set(title=layer.upper(), xlabel='Epoch')
        ax.grid(alpha=0.22)
    axes[0].set_ylabel(metric)
    axes[-1].legend(frameon=False, fontsize=7)
    fig.tight_layout()
    out = (ZOOM_DIR if zoom_start is not None else PLOT_DIR) / ((f'zoom_epoch{zoom_start}_' if zoom_start is not None else 'full_') + metric + '.png')
    fig.savefig(out, dpi=180, bbox_inches='tight')
    plt.show()
for metric, ref in [('alpha', 2.0), ('num_traps', 0.0), ('ERG_gap', 0.0), ('trace_log_midpoint_per_eval', 0.0), ('stable_rank', None)]:
    if metric in valid.columns:
        plot_spec(metric, ref=ref)
        plot_spec(metric, zoom_start=3, ref=ref)
plot_spec('alpha', zoom_start=10, ref=2.0, ylims={'fc1': (1.5, 4.5), 'fc2': (1.5, 6.0), 'fc3': (1.5, 5.0)})


In [ ]:
def fmt(arm, checkpoint, metric, scale=1.0):
    row = metric_row(terminal_summary, arm, checkpoint, metric)
    return f'{scale * float(row["mean"]):.4f} ± {scale * float(row["ci_half_width"]):.4f}'
lines = ['## Summary', '', '| Arm | Final test acc % | Final test loss | Selected test acc % | Selected test loss |', '|---|---:|---:|---:|---:|']
for arm in ARM_ORDER:
    lines.append(f'| {ARM_LABELS[arm]} | {fmt(arm, "final", "test_accuracy", 100.0)} | {fmt(arm, "final", "test_loss")} | {fmt(arm, "validation_selected", "test_accuracy", 100.0)} | {fmt(arm, "validation_selected", "test_loss")} |')
display(Markdown(chr(10).join(lines)))
